# SHAP Explainability — Fraud Detection

Covers:
- Built-in XGBoost feature importance (top 10)
- SHAP TreeExplainer: summary plot, bar plot
- SHAP Force Plots: true positive, false positive, false negative
- SHAP vs built-in importance comparison
- Business interpretation

Both datasets (Fraud_Data and CreditCard) are analyzed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
shap.initjs()

## 1. Load Models and Test Data

In [ ]:
xgb_f  = joblib.load('../models/xgb_fraud.pkl')
xgb_cc = joblib.load('../models/xgb_creditcard.pkl')

fraud_test = pd.read_csv('../data/processed/fraud_test.csv')
X_test_f   = fraud_test.drop(columns=['class'])
y_test_f   = fraud_test['class']

cc_test   = pd.read_csv('../data/processed/creditcard_test.csv')
X_test_cc = cc_test.drop(columns=['Class'])
y_test_cc = cc_test['Class']

y_pred_f  = xgb_f.predict(X_test_f)
y_pred_cc = xgb_cc.predict(X_test_cc)

print('Fraud_Data  test shape:', X_test_f.shape)
print('CreditCard  test shape:', X_test_cc.shape)

## 2. Built-in Feature Importance (Top 10)

In [ ]:
def plot_builtin_importance(model, feature_names, title, top_n=10):
    imp = pd.Series(model.feature_importances_, index=feature_names)
    imp = imp.nlargest(top_n).sort_values()
    imp.plot(kind='barh', figsize=(9, 5), title=title)
    plt.xlabel('Feature Importance (gain)')
    plt.tight_layout()
    plt.show()
    return imp

imp_f  = plot_builtin_importance(xgb_f,  X_test_f.columns,  'XGBoost Built-in Importance — Fraud_Data')
imp_cc = plot_builtin_importance(xgb_cc, X_test_cc.columns, 'XGBoost Built-in Importance — CreditCard')

## 3. SHAP TreeExplainer

In [ ]:
explainer_f  = shap.TreeExplainer(xgb_f)
explainer_cc = shap.TreeExplainer(xgb_cc)

# Use a sample for speed on large datasets
sample_f  = X_test_f.sample(min(2000, len(X_test_f)),  random_state=42)
sample_cc = X_test_cc.sample(min(2000, len(X_test_cc)), random_state=42)

shap_vals_f  = explainer_f.shap_values(sample_f)
shap_vals_cc = explainer_cc.shap_values(sample_cc)

print('SHAP values computed.')

## 4. SHAP Summary Plot — Global Feature Importance

In [ ]:
print('=== Fraud_Data — SHAP Summary (Beeswarm) ===')
shap.summary_plot(shap_vals_f, sample_f, max_display=15)

In [ ]:
print('=== Fraud_Data — SHAP Bar Plot (Mean |SHAP|) ===')
shap.summary_plot(shap_vals_f, sample_f, plot_type='bar', max_display=15)

In [ ]:
print('=== CreditCard — SHAP Summary (Beeswarm) ===')
shap.summary_plot(shap_vals_cc, sample_cc, max_display=15)

In [ ]:
print('=== CreditCard — SHAP Bar Plot (Mean |SHAP|) ===')
shap.summary_plot(shap_vals_cc, sample_cc, plot_type='bar', max_display=15)

## 5. SHAP vs Built-in Importance Comparison

In [ ]:
def compare_importance(shap_values, sample, builtin_imp, title):
    shap_mean = pd.Series(
        np.abs(shap_values).mean(axis=0),
        index=sample.columns
    ).nlargest(10)

    common = shap_mean.index.intersection(builtin_imp.index)
    comp = pd.DataFrame({
        'SHAP (mean |φ|)': shap_mean[common],
        'Built-in':        builtin_imp[common]
    })
    # Normalise to [0,1] for side-by-side comparison
    comp = comp / comp.max()
    comp.sort_values('SHAP (mean |φ|)', ascending=True).plot(
        kind='barh', figsize=(10, 5), title=f'SHAP vs Built-in — {title}'
    )
    plt.tight_layout()
    plt.show()
    return shap_mean

shap_top_f  = compare_importance(shap_vals_f,  sample_f,  imp_f,  'Fraud_Data')
shap_top_cc = compare_importance(shap_vals_cc, sample_cc, imp_cc, 'CreditCard')

## 6. SHAP Force Plots — Individual Predictions

We select three cases from the Fraud_Data test set:
- **True Positive (TP)**: actual fraud, model correctly predicted fraud
- **False Positive (FP)**: actual legit, model incorrectly predicted fraud
- **False Negative (FN)**: actual fraud, model incorrectly predicted legit

In [ ]:
y_pred_all = xgb_f.predict(X_test_f)
y_true_all = y_test_f.values

tp_idx = np.where((y_true_all == 1) & (y_pred_all == 1))[0][0]
fp_idx = np.where((y_true_all == 0) & (y_pred_all == 1))[0][0]
fn_idx = np.where((y_true_all == 1) & (y_pred_all == 0))[0][0]

print(f'True Positive  index: {tp_idx}')
print(f'False Positive index: {fp_idx}')
print(f'False Negative index: {fn_idx}')

In [ ]:
# Compute SHAP values for all test rows (needed for force plots)
shap_vals_all = explainer_f.shap_values(X_test_f)
expected_val  = explainer_f.expected_value

print('=== Force Plot: True Positive (correctly identified fraud) ===')
shap.force_plot(
    expected_val,
    shap_vals_all[tp_idx],
    X_test_f.iloc[tp_idx],
    matplotlib=True
)
plt.title('Force Plot — True Positive')
plt.tight_layout()
plt.show()

In [ ]:
print('=== Force Plot: False Positive (legitimate flagged as fraud) ===')
shap.force_plot(
    expected_val,
    shap_vals_all[fp_idx],
    X_test_f.iloc[fp_idx],
    matplotlib=True
)
plt.title('Force Plot — False Positive')
plt.tight_layout()
plt.show()

In [ ]:
print('=== Force Plot: False Negative (missed fraud) ===')
shap.force_plot(
    expected_val,
    shap_vals_all[fn_idx],
    X_test_f.iloc[fn_idx],
    matplotlib=True
)
plt.title('Force Plot — False Negative')
plt.tight_layout()
plt.show()

## 7. SHAP Dependence Plots — Top Features

In [ ]:
# Dependence plots for the top 3 SHAP features in Fraud_Data
top3_f = shap_top_f.nlargest(3).index.tolist()

for feat in top3_f:
    shap.dependence_plot(feat, shap_vals_f, sample_f, show=True)

## 8. Print Top 5 Fraud Drivers (SHAP)

In [ ]:
print('=== Top 5 Fraud Drivers — Fraud_Data (SHAP) ===')
for rank, (feat, val) in enumerate(shap_top_f.nlargest(5).items(), 1):
    print(f'{rank}. {feat:35s}  mean |SHAP| = {val:.4f}')

print('\n=== Top 5 Fraud Drivers — CreditCard (SHAP) ===')
for rank, (feat, val) in enumerate(shap_top_cc.nlargest(5).items(), 1):
    print(f'{rank}. {feat:35s}  mean |SHAP| = {val:.4f}')